In [1]:
import pandas as pd
import numpy as np
import shutil
import os
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding, 
    EarlyStoppingCallback
)

In [2]:
configuraciones = [
    {"nombre": "prep_0_len_4096", "archivo": "../Data/interim/tcga_simple_train_preprocessed_0.csv", "max_len": 4096},
    {"nombre": "prep_3_len_4096", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3.csv", "max_len": 4096},
]

In [3]:
label_mapping = {"T1": 0, "T2": 1, "T3": 2, "T4": 3}
tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

resultados_finales = []

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
for conf in configuraciones:
    print("\n" + "="*50)
    print(f"TRABAJANDO EN: {conf['nombre']}")
    print("="*50)

    df = pd.read_csv(conf['archivo'])
    X_train, X_test, y_train, y_test = train_test_split(df['text'], df['t'], test_size=0.2, random_state=23, stratify=df['t'])
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=33, stratify=y_train)

    train_ds = Dataset.from_dict({"text": X_train.tolist(), "labels": [label_mapping[label] for label in y_train.tolist()]})
    val_ds = Dataset.from_dict({"text": X_val.tolist(), "labels": [label_mapping[label] for label in y_val.tolist()]})
    test_ds = Dataset.from_dict({"text": X_test.tolist(), "labels": [label_mapping[label] for label in y_test.tolist()]})

    def tokenize_fn(df):
        return tokenizer(df["text"], padding="max_length", truncation=True, max_length=conf['max_len'])

    train_tkn = train_ds.map(tokenize_fn, batched=True)
    val_tkn = val_ds.map(tokenize_fn, batched=True)
    test_tkn = test_ds.map(tokenize_fn, batched=True)

    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    weights = compute_class_weight("balanced", classes=np.unique(y_train_encoded), y=y_train_encoded)
    weights_tensor = torch.tensor(weights, dtype=torch.float)

    model = AutoModelForSequenceClassification.from_pretrained("allenai/longformer-base-4096", num_labels=4)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.loss_function = torch.nn.CrossEntropyLoss(weight=weights_tensor.to(device))

    training_args = TrainingArguments(
        output_dir=f"../Models/longformer-{conf['nombre']}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        num_train_epochs=20,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=True if torch.cuda.is_available() else False,
        logging_steps=200,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tkn,
        eval_dataset=val_tkn,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()

    metrics = trainer.evaluate(test_tkn)
    resultados_finales.append({
        "Escenario": conf['nombre'],
        "Longitud": conf['max_len'],
        "Macro-F1": metrics['eval_macro_f1']
    })

    del model
    del trainer
    shutil.rmtree("../Models")
    os.makedirs("../Models")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


TRABAJANDO EN: prep_0_len_4096


Map:   0%|          | 0/3300 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

Map:   0%|          | 0/1032 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

[transformers] LongformerForSequenceClassification LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     | 
-------------------------------+------------+-
lm_head.bias                   | UNEXPECTED | 
lm_head.dense.bias             | UNEXPECTED | 
lm_head.layer_norm.bias        | UNEXPECTED | 
lm_head.layer_norm.weight      | UNEXPECTED | 
lm_head.decoder.weight         | UNEXPECTED | 
lm_head.dense.weight           | UNEXPECTED | 
longformer.pooler.dense.bias   | UNEXPECTED | 
longformer.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias       | MISSING    | 
classifier.dense.weight        | MISSING    | 
classifier.out_proj.weight     | MISSING    | 
classifier.dense.bias          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream ta

OutOfMemoryError: CUDA out of memory. Tried to allocate 180.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 20.38 GiB is allocated by PyTorch, and 72.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
df_results = pd.DataFrame(resultados_finales)
print("\n--- COMPARATIVA FINAL ---")
print(df_results)